In [1]:
import pathlib 
import csv 
import datetime as dt 
import time 
import shutil 




def generate_backup(input_folder, output_location): 

    now = dt.datetime.now()
    now_str = now.strftime("%Y-%m-%d_%H-%M-%S") 
    output_folder = output_location / now_str

    # Keep track of how long it takes to create the backup 
    start = time.perf_counter() 
    print("Creating back up...") 
    print(f"Input: {input_folder}") 
    print(f"Output: {output_folder}")

    # Read skip list 
    skip_files_path = pathlib.Path("C:/Users/johnm/Projects/GenerateBackup/skip_files.txt")
    skip_paths = []
    with open(skip_files_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            # remove quotes from "Copy as path"
            path = pathlib.Path(line.strip('"'))
            skip_paths.append(path)

    # Start creating backup details folder 
    output_folder.mkdir(parents=True, exist_ok=True)
    backup_details_path = output_folder / "backup_details.csv"
    with open(backup_details_path, "w", newline="", encoding="utf-8") as f:

        # Create header for backup details file 
        writer = csv.writer(f)
        writer.writerow(["input_path", "output_path", "size", "include_file"])

        # Loop over all files inside of all folders and subfolders of the input folder 
        for input_path in input_folder.rglob("*"): 

            # Skip directories; only include files 
            if input_path.is_dir(): 
                continue

            # Output file path 
            relative_path = input_path.relative_to(input_folder)
            output_path = output_folder / relative_path 
        
            # Record file sizes 
            size = input_path.stat().st_size 
            
            # Decide whether to include this file in the backup 
            include_file = True
            for path in skip_paths:
                if input_path == path or path in input_path.parents:
                    include_file = False
                    break

            # Add information to backup details file 
            writer.writerow([str(input_path), str(output_path), size, include_file]) 

            # Copy file from input_path to output_path 
            if include_file == True: 
                output_path.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(input_path, output_path)  # copy2 preserves metadata like modified time

    # End timer 
    runtime = time.perf_counter() - start 
    print(f"Runtime: {runtime:.2f} seconds") 

    # Calculate total size of input and output folders 
    input_size = 0 
    output_size = 0 
    files_included = 0 
    files_skipped = 0 
    with open(backup_details_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            input_size += int(row["size"])  
            if row["include_file"].strip().lower() == "true":
                output_size += int(row["size"])
                files_included += 1
            else:
                files_skipped += 1

    def human_readable_size(num_bytes):
        for unit in ["B", "KB", "MB", "GB", "TB"]:
            if num_bytes < 1024:
                return f"{num_bytes:.2f} {unit}"
            num_bytes /= 1024
        return f"{num_bytes:.2f} PB"

    # Save summary of backup 
    backup_summary_path = output_folder / "backup_summary.txt"
    with open(backup_summary_path, "w", encoding="utf-8") as f: 
        f.write(f"{now}\n")
        f.write(f"Time elapsed: {runtime:.1f} seconds\n")
        f.write(f"Files included: {files_included}\n") 
        f.write(f"Files skipped: {files_skipped}\n")
        f.write("\n") 
        f.write(f"{str(input_folder)}\n") 
        f.write(f"Input folder size = {human_readable_size(input_size)}\n") 
        f.write("\n") 
        f.write(f"{str(output_folder)}\n") 
        f.write(f"Output folder size = {human_readable_size(output_size)}\n") 





In [2]:
# Backup Projects folder to OneDrive 

generate_backup(
    input_folder = pathlib.Path("C:/Users/johnm/Projects"), 
    output_location = pathlib.Path("C:/Users/johnm/OneDrive/Projects_Backups") 
)



Creating back up...
Input: C:\Users\johnm\Projects
Output: C:\Users\johnm\OneDrive\Projects_Backups\2026-04-02_17-35-28
Runtime: 57.54 seconds


In [3]:
# Backup Projects folder to external hard drive 

generate_backup(
    input_folder = pathlib.Path("C:/Users/johnm/Projects"), 
    output_location = pathlib.Path("D:/Projects_Backups") 
)



Creating back up...
Input: C:\Users\johnm\Projects
Output: D:\Projects_Backups\2026-04-02_01-43-37
Runtime: 15.11 seconds
